# Visualize Preprocessed DNS Output

This notebook loads a subsampled `.npz` file (generated by `preprocess_dns_output.py`) and provides interactive tools to visualize the data. The main plot is a time-evolution diagram (Hovmöller diagram) of a selected channel along the z-axis at a specific (x, y) location.

This is a crucial step for validating the preprocessing script and gaining an initial understanding of the simulation dynamics.

### 1. Setup and Imports

In [1]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Make sure your packages are installed in editable mode
from mhd_surrogate_core.plot import plot_z_time_evolution

### 2. Load Data and Configure Widgets

**Action Required:** Change the `data_file_path` variable below to point to the specific `.npz` file you want to analyze.

In [2]:
# Example path - update this to your specific file
data_file_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/data/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325.npz")

# Load the data once to configure the interactive widgets
if not data_file_path.exists():
    print(f"ERROR: Data file not found at {data_file_path}")
else:
    with np.load(data_file_path, allow_pickle=True) as data:
        channel_options = list(data['labels'])
        max_x_index = data['timeseries'].shape[1] - 1
        max_y_index = data['timeseries'].shape[2] - 1

    # --- Create Widgets ---
    channel_dropdown = widgets.Dropdown(
        options=channel_options,
        description='Channel:',
        value=channel_options[0]
    )

    x_slider = widgets.IntSlider(
        min=0, max=max_x_index, value=max_x_index // 2, description='X Index:'
    )

    y_slider = widgets.IntSlider(
        min=0, max=max_y_index, value=max_y_index // 2, description='Y Index:'
    )

    # Link widgets to the plotting function
    interactive_plot = widgets.interactive_output(
        plot_z_time_evolution,
        {'data_path': widgets.fixed(data_file_path), 
         'channel': channel_dropdown, 
         'x_index': x_slider, 
         'y_index': y_slider}
    )

    # Display the controls and the plot output
    display(widgets.VBox([widgets.HBox([channel_dropdown, x_slider, y_slider]), interactive_plot]))